# QA: Phase Error Correction — Batch Export

For every patient in the splits file, generate QA figures at:
- **4 evenly spaced axial slices** across the full z-range
- **1 mid-coronal slice**

All images are saved to a **local folder** (e.g. `~/Desktop/qa_pec_export/`)
for upload to Google Drive.

Filename format: `{split}_{patient_id}_{plane}_{dim}{index:03d}.png`

The figure layout matches `qa_phase_error_correction.ipynb` (11 rows).

In [1]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import zoom

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"
REPO_ROOT = _PROJECT_ROOT
DOWNSAMPLED_FOLDER = "downsampled_full_fov_128x128x64"

RUN_NAME = "lucky-eon_epoch_68"
INFERENCE_DIR = WORKING_DIR.parent / "inference" / RUN_NAME

OUTPUT_DIR = Path("/Volumes/repository/vascular-superenhancement-4d-flow/data_qa") / "qa_pec" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FRAME_INDEX = 4
N_AXIAL_SLICES = 6

print(f"Patient data dir: {PATIENT_DATA_DIR}")
print(f"Inference dir:    {INFERENCE_DIR}")
print(f"Output dir:       {OUTPUT_DIR}")

Found project root at: /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow
Patient data dir: /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data
Inference dir:    /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/inference/lucky-eon_epoch_68
Output dir:       /Volumes/repository/vascular-superenhancement-4d-flow/data_qa/qa_pec/lucky-eon_epoch_68


In [2]:
splits_df = pd.read_csv(REPO_ROOT / "splits" / "splits_01-15-26.csv")

# --- Filtering options (set to None / empty to disable) ---
FILTER_SPLITS: list[str] | None = None          # e.g. ["test"]
FILTER_PATIENTS: list[str] | None = None         # e.g. ["Biswifo", "Balboloop"]

patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
if FILTER_SPLITS:
    patients_df = patients_df[patients_df["split"].isin(FILTER_SPLITS)]
if FILTER_PATIENTS:
    patients_df = patients_df[patients_df["patient_id"].isin(FILTER_PATIENTS)]
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Total patients to process: {len(patients_df)}")
print(patients_df["split"].value_counts())
if FILTER_SPLITS:
    print(f"  (filtered to splits: {FILTER_SPLITS})")
if FILTER_PATIENTS:
    print(f"  (filtered to patients: {FILTER_PATIENTS})")

Total patients to process: 215
split
train         165
test           28
validation     22
Name: count, dtype: int64


In [3]:
COMPONENTS = ["vx", "vy", "vz"]
PLANE_DIM = {"axial": 2, "coronal": 1}


def _take_slice(vol: np.ndarray, plane: str, slice_idx: int | None) -> np.ndarray:
    dim = PLANE_DIM[plane]
    if slice_idx is None:
        slice_idx = vol.shape[dim] // 2
    slicing = [slice(None)] * vol.ndim
    slicing[dim] = slice_idx
    s = vol[tuple(slicing)]
    if plane == "coronal":
        s = s[::-1, ::-1]  # flip L/R (axis 0 = x) and S/I (axis 1 = z)
    return s


def load_slice(nifti_path: Path, plane: str = "axial",
               slice_idx: int | None = None) -> np.ndarray:
    vol = nib.load(str(nifti_path)).get_fdata(dtype=np.float32)
    return _take_slice(vol, plane, slice_idx)


def _try_load_slice(nifti_path: Path, plane: str = "axial",
                    slice_idx: int | None = None) -> np.ndarray | None:
    if not nifti_path.exists():
        return None
    return load_slice(nifti_path, plane, slice_idx)


def load_mask_slice(patient_dir: Path, pid: str, ds_shape: tuple,
                    plane: str = "axial",
                    slice_idx: int | None = None) -> np.ndarray:
    mask_path = patient_dir / "nifti" / "velocity_correction" / f"correction_air_mask_{pid}.nii.gz"
    mask_vol = nib.load(str(mask_path)).get_fdata(dtype=np.float32)
    if mask_vol.shape != ds_shape:
        scale = np.array(ds_shape) / np.array(mask_vol.shape)
        mask_vol = zoom(mask_vol, scale, order=0)
    return _take_slice(mask_vol, plane, slice_idx) > 0.5


def apply_mask(arr: np.ndarray, mask: np.ndarray) -> np.ma.MaskedArray:
    return np.ma.masked_where(~mask, arr)


def _map_slice_idx(slice_idx: int, src_shape: tuple, dst_shape: tuple,
                   plane: str) -> int:
    """Map a slice index from src resolution to dst resolution along *plane* dim."""
    dim = PLANE_DIM[plane]
    src_n = src_shape[dim]
    dst_n = dst_shape[dim]
    if src_n == dst_n:
        return slice_idx
    return int(round(slice_idx / max(src_n - 1, 1) * max(dst_n - 1, 1)))


def make_qa_figure(
    pid: str, split: str, frame: int, ds_root: Path, patient_dir: Path,
    inf_root: Path | None = None,
    plane: str = "axial", slice_idx: int | None = None,
) -> plt.Figure:
    mag_path = ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz"
    mag_slice = load_slice(mag_path, plane, slice_idx)

    uncorr, corr = {}, {}
    for comp in COMPONENTS:
        uncorr[comp] = _try_load_slice(
            ds_root / f"4d_flow_{comp}" / f"4d_flow_{comp}_{pid}_frame_{frame:02d}.nii.gz",
            plane, slice_idx,
        )
        corr[comp] = _try_load_slice(
            ds_root / f"4d_flow_{comp}_corr" / f"4d_flow_{comp}_corr_{pid}_frame_{frame:02d}.nii.gz",
            plane, slice_idx,
        )

    has_uncorr = all(uncorr[c] is not None for c in COMPONENTS)
    has_corr = all(corr[c] is not None for c in COMPONENTS)
    has_diff = has_uncorr and has_corr
    diff = {comp: corr[comp] - uncorr[comp] for comp in COMPONENTS} if has_diff else {}

    # --- CNN predictions ---
    pred_raw, pred_poly = {}, {}
    ds_shape_3d = nib.load(str(mag_path)).shape
    if inf_root is not None and inf_root.exists():
        poly_slice_idx = None
        if slice_idx is not None:
            first_poly = inf_root / "predicted_corrected_velocity" / f"pred_correction_vx_{pid}.nii.gz"
            if first_poly.exists():
                poly_shape = nib.load(str(first_poly)).shape
                poly_slice_idx = _map_slice_idx(slice_idx, ds_shape_3d, poly_shape, plane)

        for comp in COMPONENTS:
            pred_raw[comp] = _try_load_slice(
                inf_root / "raw_predictions" / f"pred_correction_{comp}_t{frame:02d}.nii.gz",
                plane, slice_idx,
            )
            pred_poly[comp] = _try_load_slice(
                inf_root / "predicted_corrected_velocity" / f"pred_correction_{comp}_{pid}.nii.gz",
                plane, poly_slice_idx,
            )
    has_pred_raw = all(pred_raw.get(c) is not None for c in COMPONENTS)
    has_pred_poly = all(pred_poly.get(c) is not None for c in COMPONENTS)

    # De-normalise CNN predictions (stored as v/VENC)
    if (has_pred_raw or has_pred_poly) and inf_root is not None:
        coeff_path = inf_root / "predicted_corrected_velocity" / f"pred_poly_coefficients_{pid}.npz"
        if coeff_path.exists():
            venc = float(np.load(str(coeff_path))["venc"])
            for c in COMPONENTS:
                if pred_raw.get(c) is not None:
                    pred_raw[c] = pred_raw[c] * venc
                if pred_poly.get(c) is not None:
                    pred_poly[c] = pred_poly[c] * venc

    # CNN corrected = uncorrected + CNN polyfit
    cnn_corr = {}
    has_cnn_corr = has_uncorr and has_pred_poly
    if has_cnn_corr:
        for comp in COMPONENTS:
            poly_slice = pred_poly[comp]
            ref_shape = uncorr[comp].shape
            if poly_slice.shape != ref_shape:
                poly_slice = zoom(poly_slice,
                                  np.array(ref_shape) / np.array(poly_slice.shape), order=1)
            cnn_corr[comp] = uncorr[comp] + poly_slice

    # --- Tissue mask ---
    tissue = load_mask_slice(patient_dir, pid, ds_shape_3d, plane, slice_idx)
    mediastinum = tissue

    # --- Manual polyfit correction ---
    manual_poly = {}
    corr_dir = patient_dir / "nifti" / "velocity_correction"
    ref_shape_2d = mag_slice.shape
    man_venc_path = corr_dir / f"poly_coefficients_{pid}.npz"
    man_venc = float(np.load(str(man_venc_path))["venc"]) if man_venc_path.exists() else 1.0
    for comp in COMPONENTS:
        mp_path = corr_dir / f"ground_truth_correction_{comp}_{pid}.nii.gz"
        if mp_path.exists():
            mp_slice = load_slice(mp_path, plane, slice_idx)
            if mp_slice.shape != ref_shape_2d:
                mp_slice = zoom(mp_slice,
                                np.array(ref_shape_2d) / np.array(mp_slice.shape), order=1)
            manual_poly[comp] = mp_slice * man_venc
        else:
            manual_poly[comp] = None
    has_manual_poly = all(manual_poly[c] is not None for c in COMPONENTS)

    # Manual polyfit corrected velocity = uncorrected + manual polyfit
    man_poly_corr = {}
    has_man_poly_corr = has_uncorr and has_manual_poly
    if has_man_poly_corr:
        for comp in COMPONENTS:
            man_poly_corr[comp] = uncorr[comp] + manual_poly[comp]

    # Residual: manual polyfit − CNN polyfit on mediastinum
    corr_residual = {}
    has_corr_residual = has_manual_poly and has_pred_poly
    if has_corr_residual:
        for comp in COMPONENTS:
            cnn_slice = pred_poly[comp]
            man_slice = manual_poly[comp]
            ref_shape = man_slice.shape
            if cnn_slice.shape != ref_shape:
                cnn_slice = zoom(cnn_slice,
                                 np.array(ref_shape) / np.array(cnn_slice.shape), order=1)
            corr_residual[comp] = man_slice - cnn_slice

    # CNN improvement: |GT - uncorr| - |GT - CNN corr|
    cnn_improvement = {}
    has_cnn_improvement = has_corr and has_cnn_corr and has_uncorr
    if has_cnn_improvement:
        for comp in COMPONENTS:
            cnn_improvement[comp] = np.abs(corr[comp] - uncorr[comp]) - np.abs(corr[comp] - cnn_corr[comp])

    # --- Colour scales ---
    vel_max = 300.0
    corr_vals = []
    if has_pred_raw:
        corr_vals.extend(pred_raw[c].ravel() for c in COMPONENTS)
    if has_pred_poly:
        corr_vals.extend(pred_poly[c].ravel() for c in COMPONENTS)
    if has_diff:
        corr_vals.extend(diff[c].ravel() for c in COMPONENTS)
    if has_corr_residual:
        corr_vals.extend(corr_residual[c].ravel() for c in COMPONENTS)
    corr_max = max(np.percentile(np.abs(np.concatenate(corr_vals)), 99), 1.0) if corr_vals else 1.0

    bg_color = "0.25"
    vel_cmap = plt.cm.RdBu_r.copy(); vel_cmap.set_bad(bg_color)
    jet_cmap = plt.cm.jet.copy();     jet_cmap.set_bad(bg_color)
    resid_cmap = plt.cm.RdBu_r.copy(); resid_cmap.set_bad(bg_color)

    dim_char = "z" if plane == "axial" else "y"
    slice_label = f"{dim_char}={slice_idx}" if slice_idx is not None else f"{dim_char}=mid"

    n_rows = 11
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, n_rows * 4), constrained_layout=True)
    fig.suptitle(
        f"{pid}  [{split}]  frame {frame:02d}  {plane} {slice_label}",
        fontsize=16, fontweight="bold",
    )

    # Row 0: CNN raw correction — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        if has_pred_raw:
            im = axes[0, j].imshow(pred_raw[comp].T, origin="upper",
                                   cmap=jet_cmap, vmin=-corr_max, vmax=corr_max)
            axes[0, j].set_title(f"CNN {comp}")
        else:
            axes[0, j].text(0.5, 0.5, f"CNN {comp}\nnot available",
                            transform=axes[0, j].transAxes, ha="center", va="center",
                            fontsize=12, color="white")
            axes[0, j].set_title(f"CNN {comp} (missing)")
    if has_pred_raw:
        fig.colorbar(im, ax=axes[0, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 1: CNN polyfit correction — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        if has_pred_poly:
            im = axes[1, j].imshow(pred_poly[comp].T, origin="upper",
                                   cmap=jet_cmap, vmin=-corr_max, vmax=corr_max)
            axes[1, j].set_title(f"CNN Polyfit {comp}")
        else:
            axes[1, j].text(0.5, 0.5, f"CNN Polyfit {comp}\nnot available",
                            transform=axes[1, j].transAxes, ha="center", va="center",
                            fontsize=12, color="white")
            axes[1, j].set_title(f"CNN Polyfit {comp} (missing)")
    if has_pred_poly:
        fig.colorbar(im, ax=axes[1, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 2: Raw manual correction (corr − uncorr) — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        if has_diff:
            im = axes[2, j].imshow(diff[comp].T, origin="upper",
                                   cmap=jet_cmap, vmin=-corr_max, vmax=corr_max)
            axes[2, j].set_title(f"Man. Corr−Uncorr {comp}")
        else:
            axes[2, j].text(0.5, 0.5, f"Man. Corr−Uncorr {comp}\nnot available",
                            transform=axes[2, j].transAxes, ha="center", va="center",
                            fontsize=12, color="white")
            axes[2, j].set_title(f"Man. Corr−Uncorr {comp} (missing)")
    if has_diff:
        fig.colorbar(im, ax=axes[2, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 3: Manual polyfit correction — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        ax = axes[3, j]
        if has_manual_poly:
            im = ax.imshow(manual_poly[comp].T, origin="upper",
                           cmap=jet_cmap, vmin=-corr_max, vmax=corr_max)
            ax.set_title(f"Man. Polyfit {comp}")
        else:
            ax.text(0.5, 0.5, f"Man. Polyfit {comp}\nnot available",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=12, color="white")
            ax.set_title(f"Man. Polyfit {comp} (missing)")
    if has_manual_poly:
        fig.colorbar(im, ax=axes[3, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 4: Manual polyfit − CNN polyfit on mediastinum — RdBu_r
    for j, comp in enumerate(COMPONENTS):
        ax = axes[4, j]
        if has_corr_residual:
            im = ax.imshow(apply_mask(corr_residual[comp], mediastinum).T, origin="upper",
                           cmap=resid_cmap, vmin=-corr_max, vmax=corr_max)
            ax.set_title(f"Man.Poly−CNN Poly {comp}")
        else:
            ax.text(0.5, 0.5, f"Man.Poly−CNN Poly {comp}\nnot available",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=10, color="white")
            ax.set_title(f"Man.Poly−CNN Poly {comp} (missing)")
    if has_corr_residual:
        fig.colorbar(im, ax=axes[4, :].tolist(), fraction=0.02, pad=0.02, label="polyfit residual")

    # Row 5: CNN corrected velocity — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        if has_cnn_corr:
            im = axes[5, j].imshow(apply_mask(cnn_corr[comp], tissue).T, origin="upper",
                                   cmap=vel_cmap, vmin=-vel_max, vmax=vel_max)
            axes[5, j].set_title(f"CNN Corr. {comp}")
        else:
            axes[5, j].text(0.5, 0.5, f"CNN Corr. {comp}\nnot available",
                            transform=axes[5, j].transAxes, ha="center", va="center",
                            fontsize=12, color="white")
            axes[5, j].set_title(f"CNN Corr. {comp} (missing)")
    if has_cnn_corr:
        fig.colorbar(im, ax=axes[5, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 6: CNN improvement |GT−uncorr| − |GT−CNN| — RdBu_r, unmasked
    for j, comp in enumerate(COMPONENTS):
        ax = axes[6, j]
        if has_cnn_improvement:
            improv_max = max(np.percentile(np.abs(cnn_improvement[comp]), 99), 1.0)
            im = ax.imshow(cnn_improvement[comp].T, origin="upper",
                           cmap=resid_cmap, vmin=-improv_max, vmax=improv_max)
            ax.set_title(f"|GT−uncorr|−|GT−CNN| {comp}")
        else:
            ax.text(0.5, 0.5, f"CNN improvement {comp}\nnot available",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=10, color="white")
            ax.set_title(f"CNN improvement {comp} (missing)")
    if has_cnn_improvement:
        fig.colorbar(im, ax=axes[6, :].tolist(), fraction=0.02, pad=0.02,
                     label="+CNN helped / −CNN hurt")

    # Row 7: Manually corrected velocity (piecewise) — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        if corr[comp] is not None:
            im = axes[7, j].imshow(apply_mask(corr[comp], tissue).T, origin="upper",
                                   cmap=vel_cmap, vmin=-vel_max, vmax=vel_max)
            axes[7, j].set_title(f"Man. Corr. {comp}")
        else:
            axes[7, j].text(0.5, 0.5, f"Man. Corr. {comp}\nnot available",
                            transform=axes[7, j].transAxes, ha="center", va="center",
                            fontsize=12, color="white")
            axes[7, j].set_title(f"Man. Corr. {comp} (missing)")
    if has_corr:
        fig.colorbar(im, ax=axes[7, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 8: Manual polyfit corrected velocity — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        ax = axes[8, j]
        if has_man_poly_corr:
            im = ax.imshow(apply_mask(man_poly_corr[comp], tissue).T, origin="upper",
                           cmap=vel_cmap, vmin=-vel_max, vmax=vel_max)
            ax.set_title(f"Man. Poly Corr. {comp}")
        else:
            ax.text(0.5, 0.5, f"Man. Poly Corr. {comp}\nnot available",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=12, color="white")
            ax.set_title(f"Man. Poly Corr. {comp} (missing)")
    if has_man_poly_corr:
        fig.colorbar(im, ax=axes[8, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 9: Uncorrected velocity — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        if uncorr[comp] is not None:
            im = axes[9, j].imshow(apply_mask(uncorr[comp], tissue).T, origin="upper",
                                   cmap=vel_cmap, vmin=-vel_max, vmax=vel_max)
            axes[9, j].set_title(f"Uncorr. {comp}")
        else:
            axes[9, j].text(0.5, 0.5, f"Uncorr. {comp}\nnot available",
                            transform=axes[9, j].transAxes, ha="center", va="center",
                            fontsize=12, color="white")
            axes[9, j].set_title(f"Uncorr. {comp} (missing)")
    if has_uncorr:
        fig.colorbar(im, ax=axes[9, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 10: Magnitude | Mask | Used Pixels
    axes[10, 0].imshow(mag_slice.T, origin="upper", cmap="gray")
    axes[10, 0].set_title("Mag")
    axes[10, 1].imshow(tissue.T.astype(float), origin="upper", cmap="gray")
    axes[10, 1].set_title("Mask")
    axes[10, 2].imshow(apply_mask(mag_slice, tissue).T, origin="upper", cmap="RdBu_r")
    axes[10, 2].set_title("Used Pixels")

    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor(bg_color)

    return fig

In [4]:
errors = []
total_images = 0

for idx, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]
    patient_dir = PATIENT_DATA_DIR / pid
    ds_root = patient_dir / "nifti" / DOWNSAMPLED_FOLDER

    if not ds_root.exists():
        print(f"[SKIP] {pid}: downsampled dir not found")
        errors.append((pid, split, "missing_dir"))
        continue

    n_mag_frames = len(list((ds_root / "4d_flow_mag").glob("*.nii.gz")))
    frame = min(FRAME_INDEX, n_mag_frames - 1)

    inf_root = INFERENCE_DIR / pid

    ref_vol = nib.load(
        str(ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz")
    )
    nx, ny, nz = ref_vol.shape

    # 4 evenly spaced axial slices + 1 mid-coronal
    axial_indices = np.linspace(0, nz - 1, N_AXIAL_SLICES, dtype=int)
    coronal_idx = ny // 2

    slices = [(int(z), "axial") for z in axial_indices]
    slices.append((coronal_idx, "coronal"))

    patient_ok = True
    for si, plane in slices:
        dim_char = "z" if plane == "axial" else "y"
        out_name = f"{split}_{pid}_{plane}_{dim_char}{si:03d}.png"
        out_path = OUTPUT_DIR / out_name
        try:
            fig = make_qa_figure(
                pid, split, frame, ds_root, patient_dir,
                inf_root=inf_root, plane=plane, slice_idx=si,
            )
            fig.savefig(out_path, dpi=120, bbox_inches="tight")
            plt.close(fig)
            total_images += 1
        except Exception as e:
            print(f"[ERR]  {pid} ({split}) {plane} {dim_char}={si}: {e}")
            errors.append((pid, split, f"{plane}_{dim_char}{si}: {e}"))
            patient_ok = False

    status = "[OK]  " if patient_ok else "[WARN]"
    print(f"{status} {pid} ({split}) — {len(slices)} images")

print(f"\nDone. {total_images} images saved to {OUTPUT_DIR}")
print(f"{len(patients_df)} patients, {len(errors)} errors.")
if errors:
    print("Errors:")
    for pid, split, msg in errors:
        print(f"  {pid} ({split}): {msg}")

[OK]   Balboloop (test) — 7 images
[OK]   Biswifo (test) — 7 images
[OK]   Bomatog (test) — 7 images
[OK]   Boochuto (test) — 7 images
[OK]   Boumorim (test) — 7 images
[OK]   Bovutou (test) — 7 images
[OK]   Cadotueg (test) — 7 images
[OK]   Detodu (test) — 7 images
[OK]   Diecudey (test) — 7 images
[OK]   Diepami (test) — 7 images
[OK]   Diequipi (test) — 7 images
[OK]   Dithigog (test) — 7 images
[OK]   Dublafer (test) — 7 images
[OK]   Dujomal (test) — 7 images
[OK]   Elagieg (test) — 7 images
[OK]   Golotag (test) — 7 images
[OK]   Grequafie (test) — 7 images
[OK]   Gueshifa (test) — 7 images
[OK]   Kuquelok (test) — 7 images
[OK]   Oduskueb (test) — 7 images
[OK]   Quetode (test) — 7 images
[OK]   Runusath (test) — 7 images
[OK]   Sepigoo (test) — 7 images
[OK]   Stonscuetof (test) — 7 images
[OK]   Suquepog (test) — 7 images
[OK]   Tercippun (test) — 7 images
[OK]   Tiepolem (test) — 7 images
[OK]   Tisupey (test) — 7 images
[OK]   Amifer (train) — 7 images
[OK]   Aruborn (train